In [ ]:
# ============================================================
# Cell 0: 硬件自检 —— 本章是概念章，CPU 即可，但顺手看一眼 device
# ============================================================
# 训练 / 推理大模型时离不开 GPU；本章只演示张量与 autograd，CPU 也能完整跑通。
# device 这一节会涉及 .to('cuda')，没有 GPU 的话相关 cell 会跳过。
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Total VRAM: {total_mem:.1f} GB')
else:
    print('未检测到 GPU —— 本章是概念章，CPU 完全够用，继续执行即可。')

In [ ]:
# ============================================================
# Cell 1: 张量的四个核心属性 —— shape / dtype / device / requires_grad
# ============================================================
# 99% 的 PyTorch 报错都来自其中之一不匹配。先看一眼一个 tensor 的全部 "身份证字段"。
import torch

x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

print('x =')
print(x)
print('shape          :', x.shape)            # torch.Size([2, 3])
print('ndim (rank)    :', x.ndim)             # 2 —— 维度数
print('dtype          :', x.dtype)            # 默认 float32（因为构造时给的是带小数点的数）
print('device         :', x.device)           # cpu
print('requires_grad  :', x.requires_grad)    # 默认 False —— 普通张量不参与 autograd
print('numel (元素总数):', x.numel())         # 6 = 2 × 3

In [ ]:
# ============================================================
# Cell 2: dtype 不一致是新手最常见的报错来源
# ============================================================
# 整数 tensor 与浮点 tensor 直接相加会自动 promotion；但矩阵乘要求一致。
import torch

a = torch.tensor([1, 2, 3])             # 默认 int64（torch.long）
b = torch.tensor([1.0, 2.0, 3.0])       # 默认 float32

print('a.dtype =', a.dtype, '  b.dtype =', b.dtype)

# 加法：PyTorch 把 a 自动提升到 float32 再算，结果是 float32
c = a + b
print('a + b =', c, '  dtype =', c.dtype)

# 显式 dtype 转换
print('a.float()  →', a.float().dtype)
print('b.long()   →', b.long().dtype)
print('b.to(torch.bfloat16) →', b.to(torch.bfloat16).dtype)

In [ ]:
# ============================================================
# Cell 3: device —— 两个张量必须在同一设备才能做运算
# ============================================================
# 没 GPU 时把这一节当作示意；有 GPU 的话能看到 cpu / cuda:0 之间搬数据。
import torch

x_cpu = torch.tensor([1.0, 2.0])
print('x_cpu.device =', x_cpu.device)

if torch.cuda.is_available():
    # 推荐用 .to(device) 而不是 .cuda()，更通用（同一份代码能跑 cpu/cuda/mps）
    device = torch.device('cuda')
    y_gpu = torch.tensor([3.0, 4.0]).to(device)
    print('y_gpu.device =', y_gpu.device)

    # ❌ 直接 x_cpu + y_gpu 会报 RuntimeError
    # ✅ 把 x 搬到同设备再算
    z = x_cpu.to(y_gpu.device) + y_gpu
    print('z =', z, '  device =', z.device)
else:
    print('（无 GPU 环境，跳过 device 演示——真实训练 / 推理代码里这一步几乎是必备）')

In [ ]:
# ============================================================
# Cell 4: 创建张量的常用方式
# ============================================================
import torch

# 固定 seed 让随机张量的输出可复现 —— 写实验对比代码时几乎都要先固定 seed
torch.manual_seed(0)

print('torch.zeros(2, 3)        →')
print(torch.zeros(2, 3))

print('\ntorch.ones(2, 3)         →')
print(torch.ones(2, 3))

print('\ntorch.arange(0, 10, 2)   →', torch.arange(0, 10, 2))
print('torch.linspace(0,1,5)    →', torch.linspace(0.0, 1.0, 5))

# 下面三个随机 API 共同的语义：每个元素都是独立同分布（i.i.d.）地从给定分布里采样，
# 不是"每一行"或"整体"满足该分布。
print('\ntorch.rand(2, 3)  (每个元素独立采样自均匀分布 [0,1)) →')
print(torch.rand(2, 3))

print('\ntorch.randn(2, 3) (每个元素独立采样自标准正态 N(0,1)) →')
print(torch.randn(2, 3))

print('\ntorch.randint(0, 100, (2, 3)) (每个元素独立采样自整数区间 [0,100)) →')
print(torch.randint(0, 100, (2, 3)))

# zeros_like / ones_like / randn_like：复用已有张量的 shape/dtype/device
x = torch.randn(2, 3)
print('\nzeros_like(x)            →')
print(torch.zeros_like(x))

In [ ]:
# ============================================================
# Cell 5: 形状操作 —— reshape / transpose / permute / squeeze
# ============================================================
# 模型 forward 里大量出现这些 API；多花 1 分钟看清楚每一步的 shape，后面读 attention 实现能少走很多弯路。
import torch

x = torch.arange(12)                    # shape (12,)
print('x shape    :', x.shape)

y = x.reshape(3, 4)                     # (12,) → (3, 4)
print('reshape(3,4):', y.shape)
print(y)

y2 = x.reshape(-1, 4)                   # -1 让 PyTorch 自动算这一维的长度
print('reshape(-1,4):', y2.shape)       # 仍然是 (3, 4)

# transpose：交换两个维度
a = torch.randn(2, 3, 4)                # (B, L, D) 风格
print('a               :', a.shape)
print('a.transpose(1,2):', a.transpose(1, 2).shape)   # (2, 4, 3)

# permute：一次性指定全部维度顺序
print('a.permute(2,0,1):', a.permute(2, 0, 1).shape)  # (4, 2, 3)

# squeeze / unsqueeze：删掉 / 插入长度 1 的维度
# 注意：squeeze(dim) 只有在该维长度为 1 时才删，否则原样返回——不会报错
u = torch.randn(1, 3, 1)
print('u shape          :', u.shape)
print('u.squeeze()      :', u.squeeze().shape)        # (3,)
print('u.squeeze(0)     :', u.squeeze(0).shape)       # (3, 1)
print('u.squeeze(1)     :', u.squeeze(1).shape)       # (1, 3, 1) —— dim 1 长度是 3，原样返回
v = torch.randn(3)
print('v.unsqueeze(0)   :', v.unsqueeze(0).shape)     # (1, 3)
print('v.unsqueeze(-1)  :', v.unsqueeze(-1).shape)    # (3, 1)

In [ ]:
# ============================================================
# Cell 5b: expand vs repeat —— 显式扩展长度 1 的维度
# ============================================================
# expand：只改 stride（步长），不拷贝内存；只能扩展长度 1 的维度，返回只读视图
# repeat：真复制；任意维度都能重复，但内存代价大
# 优先用 expand，需要写入或源维不是 1 时才用 repeat
import torch

x = torch.tensor([[1., 2., 3.]])      # shape (1, 3)

print('x.expand(4, 3) =')
print(x.expand(4, 3))                 # 4 行都是 [1,2,3]
print('expand 后 stride :', x.expand(4, 3).stride())  # (0, 1) —— dim 0 stride=0，4 行共享内存

print('\nx.repeat(4, 1) =')
print(x.repeat(4, 1))                 # 输出一样，但真分配了 4 份内存

# expand 只能扩展长度 1 的维度
y = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])      # shape (2, 3)

try:
    y.expand(4, 3)                    # dim 0 长度是 2，不是 1 —— 报错
except RuntimeError as e:
    print('\n❌ y.expand(4, 3) 报错:', str(e).splitlines()[0])

print('\ny.repeat(2, 1) =')
print(y.repeat(2, 1))                 # ✅ (4, 3)：在 dim 0 上把 y 整体重复 2 次

# 参数语义对比：expand(*sizes) 写"目标形状"（-1 表示该维不变）；repeat(*times) 写"每维重复几次"
print('\nx.expand(5, -1) =', x.expand(5, -1).shape)   # (5, 3) —— dim 0 从 1 扩到 5，dim 1 保持 3
print('x.repeat(5, 2)  =', x.repeat(5, 2).shape)       # (5, 6) —— dim 0 重复 5 次，dim 1 重复 2 次

In [ ]:
# ============================================================
# Cell 6: 模拟 multi-head attention 的形状变换
# ============================================================
# Transformer 的 multi-head attention 里有个核心形状变换：
#   (B, L, D) → (B, L, H, D/H) → (B, H, L, D/H)
# 提前在这里跑一遍建立直觉；具体语义（为什么要分头、Q/K/V 是什么）留到实际学 attention 时再讲。
import torch

B, L, D, H = 2, 5, 12, 4                # batch=2, seq=5, hidden=12, heads=4
x = torch.randn(B, L, D)                # 假装这是 attention 入口的张量
print('入口 x        :', x.shape)

# 把 hidden_dim 拆成 (H, D/H) ——一次 reshape 完成
x_split = x.reshape(B, L, H, D // H)
print('split heads   :', x_split.shape)   # (2, 5, 4, 3)

# 交换 L 和 H 两个维度，让 head 维到 batch 之后，方便后续按 head 做矩阵乘
x_heads = x_split.transpose(1, 2)
print('transpose(1,2):', x_heads.shape)   # (2, 4, 5, 3)

# 上面这两步等价于 permute(0, 2, 1, 3)
x_heads2 = x.reshape(B, L, H, D // H).permute(0, 2, 1, 3)
print('permute equiv :', x_heads2.shape)

# 验证两条路径结果完全一样
print('两种写法是否一致 :', torch.equal(x_heads, x_heads2))

In [ ]:
# ============================================================
# Cell 7: Broadcasting —— 维度自动对齐
# ============================================================
# 规则：从最右侧维度对齐，遇到"长度相等" / "一边是 1" / "一边没有这一维"都合法，否则报错。
import torch

# 例 1：(2, 3, 4) + (3, 4) —— 后者左侧自动补 1 → (1, 3, 4) → 广播到 (2, 3, 4)
A = torch.ones(2, 3, 4)
B = torch.ones(   3, 4) * 10
print('(2,3,4) + (3,4) →', (A + B).shape)

# 例 2：给 batch 里每条样本加一个 bias 向量 —— LLM 里非常常见
bias = torch.randn(4)                # shape (4,)
out = torch.randn(2, 3, 4) + bias    # (2,3,4) + (4,) → (2,3,4)
print('给每个位置加同一个 bias 向量:', out.shape)

# 例 3：按 batch 提供独立的 scale
scale = torch.tensor([[1.0], [2.0]]).unsqueeze(-1)  # shape (2, 1, 1)
scaled = torch.ones(2, 3, 4) * scale                # 每个 batch 各自乘一个标量
print('按 batch 独立缩放        :', scaled.shape)
print('第 0 个 batch 全 1，第 1 个全 2 ：')
print(scaled)

# 例 4：故意触发 broadcasting 失败
try:
    bad = torch.ones(2, 3, 4) + torch.ones(2, 5, 4)   # 中间维 3 vs 5，不可广播
except RuntimeError as e:
    print('\n❌ 不可广播报错示例:')
    print(' ', str(e).splitlines()[0])

In [ ]:
# ============================================================
# Cell 8: 逐元素运算 vs 矩阵乘
# ============================================================
# `*` 是逐元素乘，不是矩阵乘 —— 这是新手最容易栽的坑。
# 矩阵乘 / 点积要写成 `@` 或 `torch.matmul`。
import torch

a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print('a * b  (逐元素相乘) =', a * b)             # [10, 40, 90]
print('a @ b  (1D 点积)   =', a @ b)              # 10 + 40 + 90 = 140

# 2D × 2D = 矩阵乘
A = torch.randn(2, 3)
B = torch.randn(3, 4)
print('(2,3) @ (3,4) →', (A @ B).shape)            # (2, 4)

# 3D × 3D = batch 矩阵乘（前面的 batch 维走广播，最后两维当矩阵）
X = torch.randn(2, 3, 4)
Y = torch.randn(2, 4, 5)
print('(2,3,4) @ (2,4,5) →', (X @ Y).shape)        # (2, 3, 5)

In [ ]:
# ============================================================
# Cell 9: Autograd 入门 —— 用 PyTorch 验证手算偏导
# ============================================================
# f(x) = (2x + 3)^2  →  f'(x) = 4(2x + 3)
# 代入 x=1: f(1) = 25, f'(1) = 20
import torch

# requires_grad=True 让 PyTorch 把 x 标记为"需要追踪梯度的叶子节点"
x = torch.tensor(1.0, requires_grad=True)

# forward —— 这一步 PyTorch 会动态构建计算图
y = 2 * x + 3                        # 中间张量
loss = y ** 2                        # 标量

# backward —— 沿计算图回溯，把 ∂loss/∂x 累加到 x.grad
loss.backward()

print('loss          =', loss.item())   # 25.0
print('x.grad        =', x.grad)        # tensor(20.) —— 与手算一致 ✅
print('y.requires_grad =', y.requires_grad, ' （True，因为依赖 x）')
print('y.grad        =', y.grad,
      '  （None —— 中间张量默认不保留梯度，要保留需先 y.retain_grad()）')

In [ ]:
# ============================================================
# Cell 10: 多变量梯度 —— 把上一节扩展到向量
# ============================================================
# 设 w = [w0, w1]，x = [1, 2]，loss = (w · x)^2
# ∂loss/∂w = 2 (w·x) · x
# 取 w = [1, 1]，则 w·x = 3，∂loss/∂w = 6 · [1, 2] = [6, 12]
import torch

w = torch.tensor([1.0, 1.0], requires_grad=True)
x = torch.tensor([1.0, 2.0])         # 输入不需要梯度

loss = (w @ x) ** 2                  # 标量
loss.backward()

print('loss   =', loss.item())       # 9.0
print('w.grad =', w.grad)            # tensor([ 6., 12.]) —— 与手算一致 ✅

In [ ]:
# ============================================================
# Cell 11: torch.no_grad / detach —— 关闭梯度的两种方式
# ============================================================
import torch

x = torch.tensor(1.0, requires_grad=True)

# 方式 1：torch.no_grad() 上下文 —— 整段 forward 都不构建计算图
with torch.no_grad():
    y = 2 * x + 3
    print('在 no_grad 中, y.requires_grad =', y.requires_grad)
    # y.backward() ❌ 会报错：no_grad 模式下 y 不是计算图节点

# 方式 2：detach() —— 只针对单个张量
y = 2 * x + 3                        # 在 autograd 范围内构造
y_static = y.detach()                # 与 y 共享数据，但脱离计算图
print('y.requires_grad         =', y.requires_grad)         # True
print('y_static.requires_grad  =', y_static.requires_grad)  # False

# 实战常用：保存 loss 数值用于打印 / 写日志，避免拉住整张图
loss = (y ** 2).mean()
loss_value = loss.detach().item()    # 取出 Python float，与 autograd 彻底解耦
print('loss_value =', loss_value)

In [ ]:
# ============================================================
# Cell 12: 梯度累加陷阱 —— 不调 zero_grad 会发生什么
# ============================================================
# PyTorch 默认 .grad 是"累加"（+=）的；连续两次 backward 不清零，梯度会翻倍。
import torch

x = torch.tensor(1.0, requires_grad=True)

# 第一次 forward+backward
loss1 = (2 * x + 3) ** 2
loss1.backward()
print('第 1 次 backward 后 x.grad =', x.grad)   # 20.0

# 第二次：不清零直接 backward
loss2 = (2 * x + 3) ** 2
loss2.backward()
print('第 2 次 backward 后 x.grad =', x.grad)   # 40.0 —— 上一次的 20 没清，被加了一遍

# ✅ 正确做法：每个 batch 开头清零
x.grad = None                           # 也可以 x.grad.zero_()，效果相同
loss3 = (2 * x + 3) ** 2
loss3.backward()
print('清零后再 backward, x.grad =', x.grad)    # 20.0 —— 干净的单次梯度

# 在真实训练循环里，这一步通过 optimizer.zero_grad() 完成 —— P02 会用上

In [ ]:
# ============================================================
# Cell 13: 综合练习 —— 用 autograd 算 sigmoid 的导数
# ============================================================
# sigmoid(x) = 1 / (1 + exp(-x))
# 它的导数有一个漂亮的恒等式：sigmoid'(x) = sigmoid(x) * (1 - sigmoid(x))
# 我们用 autograd 在多个点上验证一遍。
import torch

# linspace 在 [-3, 3] 上等距取 7 个点；requires_grad=True 让每一项都进计算图
xs = torch.linspace(-3.0, 3.0, 7, requires_grad=True)
ys = torch.sigmoid(xs)

# 注意：xs 是向量，ys 也是向量，不能直接 ys.backward()。
# 想拿到 dy_i/dx_i 的逐点导数，最简单的办法是 ys.sum().backward()：
# 因为 ∂(Σ y_j)/∂x_i = ∂y_i/∂x_i（其它项与 x_i 无关），结果就是逐点导数。
ys.sum().backward()

# autograd 算出的导数
auto_grad = xs.grad
# 用恒等式手算的"理论导数"
with torch.no_grad():
    expected = torch.sigmoid(xs) * (1 - torch.sigmoid(xs))

print(f"{'x':>6} {'sigmoid(x)':>12} {'autograd':>12} {'expected':>12}")
for xi, yi, gi, ei in zip(xs.detach(), ys.detach(), auto_grad, expected):
    print(f'{xi.item():6.2f} {yi.item():12.4f} {gi.item():12.4f} {ei.item():12.4f}')

# 两列应当完全一致（浮点误差可忽略）